In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import os
import numpy as np
import yfinance as yf


In [2]:
def fetch_info():
    try:
        url = f"https://en.wikipedia.org/wiki/List_of_Dow_Jones_Industrial_Average_companies"
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36,accept-language=en-US,en;q=0.9,accept:application/json"
        }
        # send get request to the Wikipedia page
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, "html.parser")
        # get symbols of Dow Jones components
        tables = soup.find_all("table")
        # convert table to DataFrame
        df = pd.read_html(str(tables))[0]
        # clean up
        df.drop(columns=['Notes'], inplace=True)
        return df

    except Exception as e:
        print(
            f"Error fetching info for Dow Jones Industrial Average from WIKI: {e}")
        return None

In [3]:
djia_components = fetch_info()
print(djia_components)
tickers = djia_components['Symbol'].tolist()
print(tickers)

                   Company Exchange Symbol                  Sector  Date added
0                       3M     NYSE    MMM             Industrials  1976-08-09
1       Alphabet (Class A)   NASDAQ  GOOGL  Communication Services  2026-06-29
2         American Express     NYSE    AXP              Financials  1982-08-30
3                    Amgen   NASDAQ   AMGN             Health Care  2020-08-31
4                   Amazon   NASDAQ   AMZN  Consumer Discretionary  2024-02-26
5                    Apple   NASDAQ   AAPL  Information Technology  2015-03-19
6                   Boeing     NYSE     BA             Industrials  1987-03-12
7              Caterpillar     NYSE    CAT             Industrials  1991-05-06
8                  Chevron     NYSE    CVX                  Energy  2008-02-19
9                    Cisco   NASDAQ   CSCO  Information Technology  2009-06-08
10               Coca-Cola     NYSE     KO        Consumer Staples  1987-03-12
11                  Disney     NYSE    DIS  Communic

/var/folders/j6/2sp53bgs4612lcxv_stnsxk00000gn/T/ipykernel_60630/672945357.py:13: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(tables))[0]


In [4]:
# fetch historical data for the tickers
start_date = "2025-01-01"
end_date = "2025-12-31"
historical_data = yf.download(
    tickers, start=start_date, end=end_date, multi_level_index=False)
historical_data = historical_data['Close']
print(historical_data)

[*********************100%***********************]  30 of 30 completed

Ticker            AAPL        AMGN        AMZN         AXP          BA  \
Date                                                                     
2025-01-02  242.093124  246.071320  220.220001  292.864929  171.869995   
2025-01-03  241.606674  247.912506  224.190002  298.127502  169.899994   
2025-01-06  243.234833  245.416458  227.610001  296.947174  170.779999   
2025-01-07  240.464951  248.709702  222.110001  297.084869  172.509995   
2025-01-08  240.951385  250.750137  222.130005  297.901276  171.759995   
...                ...         ...         ...         ...         ...   
2025-12-23  271.620667  324.843079  232.139999  379.103363  216.850006   
2025-12-24  273.066742  327.263550  232.380005  380.015930  218.160004   
2025-12-26  272.657806  326.254181  232.520004  377.972565  216.440002   
2025-12-29  273.016876  323.020416  232.070007  372.288818  217.250000   
2025-12-30  272.338684  322.099213  232.529999  370.374451  218.500000   

Ticker             CAT         CRM   

In [5]:
monthly_returns = historical_data.resample('M').ffill().pct_change()
print(monthly_returns)

Ticker          AAPL      AMGN      AMZN       AXP        BA       CAT  \
Date                                                                     
2025-01-31       NaN       NaN       NaN       NaN       NaN       NaN   
2025-02-28  0.025873  0.088042 -0.106866 -0.051945 -0.010707 -0.074009   
2025-03-31 -0.081500  0.011329 -0.103731 -0.106027 -0.023364 -0.041140   
2025-04-30 -0.043353 -0.066217 -0.030695 -0.006525  0.074406 -0.057735   
2025-05-31 -0.053584 -0.000664  0.111647  0.103750  0.131412  0.125328   
2025-06-30  0.021509 -0.031126  0.070143  0.084782  0.010660  0.115450   
2025-07-31  0.011698  0.056910  0.067095 -0.059296  0.058751  0.132440   
2025-08-31  0.119639 -0.017133 -0.021827  0.106812  0.057880 -0.043331   
2025-09-30  0.096881 -0.019151 -0.041179  0.002656 -0.080322  0.138674   
2025-10-31  0.061815  0.057512  0.112265  0.088765 -0.068619  0.213284   
2025-11-30  0.032364  0.165848 -0.045041  0.012585 -0.059795 -0.002599   
2025-12-31 -0.020692 -0.048544 -0.0029

/var/folders/j6/2sp53bgs4612lcxv_stnsxk00000gn/T/ipykernel_60630/1568143435.py:1: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_returns = historical_data.resample('M').ffill().pct_change()


In [ ]:
#see the difference between simple monthly returns and compounded monthly returns
mth_return_df = historical_data.pct_change().resample('M').agg(lambda x: (1 + x).prod() - 1)
print(mth_return_df)

Ticker          AAPL      AMGN      AMZN       AXP        BA       CAT  \
Date                                                                     
2025-01-31 -0.032192  0.100818  0.079284  0.066235  0.027055  0.036222   
2025-02-28  0.025873  0.088042 -0.106866 -0.051945 -0.010707 -0.074009   
2025-03-31 -0.081500  0.011329 -0.103731 -0.106027 -0.023364 -0.041140   
2025-04-30 -0.043353 -0.066217 -0.030695 -0.006525  0.074406 -0.057735   
2025-05-31 -0.053584 -0.000664  0.111647  0.103750  0.131412  0.125328   
2025-06-30  0.021509 -0.031126  0.070143  0.084782  0.010660  0.115450   
2025-07-31  0.011698  0.056910  0.067095 -0.059296  0.058751  0.132440   
2025-08-31  0.119639 -0.017133 -0.021827  0.106812  0.057880 -0.043331   
2025-09-30  0.096881 -0.019151 -0.041179  0.002656 -0.080322  0.138674   
2025-10-31  0.061815  0.057512  0.112265  0.088765 -0.068619  0.213284   
2025-11-30  0.032364  0.165848 -0.045041  0.012585 -0.059795 -0.002599   
2025-12-31 -0.020692 -0.048544 -0.0029

/var/folders/j6/2sp53bgs4612lcxv_stnsxk00000gn/T/ipykernel_60630/2249956258.py:1: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  mth_return_df = historical_data.pct_change().resample('M').agg(lambda x: (1 + x).prod() - 1)


In [7]:
#calculate historical cumulative returns of past 6 months
cumulative_returns_df = (mth_return_df + 1).rolling(window=6).apply(np.prod) - 1
print(cumulative_returns_df)

Ticker          AAPL      AMGN      AMZN       AXP        BA       CAT  \
Date                                                                     
2025-01-31       NaN       NaN       NaN       NaN       NaN       NaN   
2025-02-28       NaN       NaN       NaN       NaN       NaN       NaN   
2025-03-31       NaN       NaN       NaN       NaN       NaN       NaN   
2025-04-30       NaN       NaN       NaN       NaN       NaN       NaN   
2025-05-31       NaN       NaN       NaN       NaN       NaN       NaN   
2025-06-30 -0.156593  0.095160 -0.003769  0.074930  0.219119  0.088221   
2025-07-31 -0.118344  0.051479 -0.015020 -0.051624  0.256741  0.189267   
2025-08-31 -0.037760 -0.050162  0.078764  0.107187  0.343870  0.228667   
2025-09-30  0.149116 -0.078788  0.154052  0.241792  0.265494  0.459078   
2025-10-31  0.275444  0.043275  0.324260  0.360899  0.097031  0.878744   
2025-11-30  0.391273  0.217109  0.137603  0.248495 -0.088366  0.665171   
2025-12-31  0.333796  0.195229  0.0598

In [ ]:
# 6-Month Cumulative Returns at the End of the Measurement Period
import datetime as dt
end_of_measurement_period = dt.datetime(2025, 10, 31)
formation_period =  dt.date(2025, 11, 30)

end_of_measurement_period_return_df = cumulative_returns_df.loc[end_of_measurement_period]
end_of_measurement_period_return_df = end_of_measurement_period_return_df.reset_index()
end_of_measurement_period_return_df.head()

,Ticker,2025-10-31 00:00:00
0,AAPL,0.275444
1,AMGN,0.043275
2,AMZN,0.324260
3,AXP,0.360899
4,BA,0.097031


In [12]:
#highest momentum stock at the end of the measurement period
highest_momentum_stock = end_of_measurement_period_return_df.loc[end_of_measurement_period_return_df.iloc[:,1].idxmax()]
highest_momentum_stock.head()

Ticker                      CAT
2025-10-31 00:00:00    0.878744
Name: 5, dtype: object

In [13]:
# Display the highest momentum stock at the end of the measurement period in negative direction
lowest_momentum_stock = end_of_measurement_period_return_df.loc[end_of_measurement_period_return_df.iloc[:,1].idxmin()]
lowest_momentum_stock.head()

Ticker                      UNH
2025-10-31 00:00:00   -0.158675
Name: 27, dtype: object

In [14]:
#using pd.qcut to categorize stocks into quantiles based on their momentum
end_of_measurement_period_return_df['rank'] = pd.qcut(end_of_measurement_period_return_df.iloc[:,1], q=5, labels=False)
end_of_measurement_period_return_df.head()

,Ticker,2025-10-31 00:00:00,rank
0,AAPL,0.275444,3
1,AMGN,0.043275,1
2,AMZN,0.324260,4
3,AXP,0.360899,4
4,BA,0.097031,2


In [42]:
#obtain stocks for long
long_stocks = end_of_measurement_period_return_df[end_of_measurement_period_return_df['rank'] == 
                                                  end_of_measurement_period_return_df['rank'].max()].values
long_stocks = long_stocks[:, 0]
long_stocks

array(['AMZN', 'AXP', 'CAT', 'GOOGL', 'GS', 'NVDA'], dtype=object)

In [43]:
#obtain stocks for short
short_stocks = end_of_measurement_period_return_df[end_of_measurement_period_return_df['rank'] == 
                                                   end_of_measurement_period_return_df['rank'].min()].values
short_stocks = short_stocks[:, 0]
short_stocks

array(['CRM', 'KO', 'MCD', 'PG', 'SHW', 'UNH'], dtype=object)

In [45]:
from dateutil.relativedelta import relativedelta
december_end = pd.Timestamp(formation_period) + pd.offsets.MonthEnd(1)
long_return_df = mth_return_df.loc[december_end, long_stocks]
#long_return_df = mth_return_df.loc[formation_period + relativedelta(months=1), mth_return_df.columns.isin(long_stocks)]
long_return_df

Ticker
AMZN    -0.002959
AXP      0.022230
CAT      0.002831
GOOGL   -0.019129
GS       0.075982
NVDA     0.059607
Name: 2025-12-31 00:00:00, dtype: float64

In [47]:
#short stocks return 
short_return_df = mth_return_df.loc[december_end, short_stocks]
short_return_df

Ticker
CRM    0.155328
KO    -0.034981
MCD   -0.006227
PG    -0.027740
SHW   -0.051005
UNH    0.014020
Name: 2025-12-31 00:00:00, dtype: float64

In [ ]:
#calculate monthly portfolio return
total_return_df = long_return_df.mean() - short_return_df.mean()
np.round(total_return_df * 100, 2)

np.float64(1.49)

In [55]:
#compare with Buy and Hold strategy for the Index DJI
df_dji = yf.download('^DJI', start=start_date, end=end_date,multi_level_index=False)

buy_n_hold_df = df_dji['Close'].pct_change().resample('M').agg(lambda x: (1 + x).prod() - 1)
buy_n_hold_df

[*********************100%***********************]  1 of 1 completed
/var/folders/j6/2sp53bgs4612lcxv_stnsxk00000gn/T/ipykernel_60630/4085773526.py:4: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  buy_n_hold_df = df_dji['Close'].pct_change().resample('M').agg(lambda x: (1 + x).prod() - 1)


Date
2025-01-31    0.050773
2025-02-28   -0.015799
2025-03-31   -0.041951
2025-04-30   -0.031723
2025-05-31    0.039359
2025-06-30    0.043168
2025-07-31    0.000821
2025-08-31    0.032039
2025-09-30    0.018729
2025-10-31    0.025108
2025-11-30    0.003228
2025-12-31    0.013635
Freq: ME, Name: Close, dtype: float64

In [58]:
#monthly return for evaluation
np.round(buy_n_hold_df.loc[december_end,] * 100, 2)

np.float64(1.36)